#TOPIK 3

In [ ]:
#1. MODUL UTILITAS (FUNGSI & MODULARISASI)

    #CSV adalah format file teks sederhana yang menyimpan data dalam bentuk tabel (baris dan kolom)
    #OS adalah modul yang menjadi jembatan penghubung antara program Python
import csv
import os

    #Buat path (jalur) folder yang mengarah ke Google Drive sesuai struktur panduan.
folderdata = f"/content/drive/MyDrive/Kelompok_3/data"

    #Mengecek apakah folder Kelompok_[No]/data sudah ada di Drive. Jika belum, buat otomatis
    #os.makedirs digunakan untuk membuat sebuah folder dan folder utama secara otomatis sekaligus jika folder tersebut belum ada
    #path.exists digunakan untuk memeriksa apakah sebuah file atau folder ada pada jalur tertentu
if not os.path.exists(folderdata):
    os.makedirs(folderdata)

    #Update variabel filecsv dan filelaporan agar mengarah ke folder Google Drive tersebut
filecsv = f"{folderdata}/data_latihan.csv"
filelaporan = f"{folderdata}/laporan_ringkasan.txt"

    #Def digunakan untuk membuat fungsi yang bisa dipanggil berulang kali
def hitungimt(beratkg, tinggicm):
    """MENGHITUNG INDEKS MASA TUBUH (IMT)"""
        #Hitung IMT (ubah tinggi satuan cm ke dalam m)
    tinggim = tinggicm / 100
        #return digunakan untuk mengembalikan nilai hasil perhitungan ke pemanggil fungsi
    return beratkg / (tinggim ** 2)

def kimt(imt):
    #menggunakan percabangan if, elif, else untuk menentukan kondisi imt atlet
    if imt < 18.5:
        return "Underweight"
    elif imt <= 24.9:
        return "Normal"
    elif imt <= 29.9:
        return "Overweight"
    else:
        return "Obesitas"

def hitunghrmax(usia):
    """Menghitung detak jantung maksimum(HRMax)"""
    return 220 - usia

def zonalatihan(hrmax):
    """Menghitung zona latihan menggunakan persentase"""
        #mengembalikan data dalam bentuk dictionary
    return {"ringan": hrmax * 0.50, "sedang": hrmax * 0.70, "berat": hrmax * 0.85}

    #intesitas="sedang" adalah default paramater, dipakai jika user tidak mengisi intensitas
def hitungkalori(beratkg, durasi, intensitas="sedang"):
    """ Menghitung perkiraan kalori yang terbakar"""
        #dictionary untuk menyimpan nilai MET
    met = {"ringan": 3.5, "sedang": 7.0, "berat": 10.5}
        #fungsi .get() untuk mengambil nilai dari dictionary, ubah input ke huruf kecil menggunakan .lower()
        #angka 7.0 berfungsi sebagai nilai default, digunakan jika intensitas tidak ditemukan di dictionary met.
    nilaimet = met.get(intensitas.lower(), 7.0)
    return nilaimet * beratkg * (durasi / 60)

def evaluasiperforma(hristirahat, sistolik, diastolik, vo2max, imt):
    """Mengevaluasi performa atlet secara keseluruhan"""
        #menggunakan percabangan if, elif, dan else untuk menentukan kondisi heart rate
    if hristirahat < 60:
        sht = "Bradikardia"
    elif hristirahat <= 100:
        sht = "Normal"
    else:
        sht = "Takikardia"

        #menggunakan percabangan if, elif, dan else untuk menentukan kondisi tekanan darah menggunakan logika AND & OR
        #AND menjadi true ketika 2 variabel terpenuhi, sedangkan OR hanya membutuh salah satu saja
    if sistolik < 120 and diastolik < 80:
        std = "Normal"
    elif sistolik < 139 and diastolik < 89:
        std = "Prehipertensi"
    elif sistolik < 159 and diastolik < 99:
        std = "Hipertensi Level 1"
    else:
        std = "Hipertensi Level 2"

        #menggunakan percabangan if, elif, dan else untuk menentukan kondisi VO2Max
    if vo2max < 33.0:
        svo = "Sangat Buruk"
    elif vo2max <= 36.4:
        svo = "Buruk"
    elif vo2max <= 42.4:
        svo = "Cukup"
    elif vo2max <= 46.4:
        svo = "Baik"
    elif vo2max <= 52.4:
        svo = "Sangat Baik"
    else:
        svo = "Superior"

        #mengembalikan beberapa variabel sekaligus dalam bentuk dictionary
    return {"hrstatus": sht, "tdstatus": std, "vo2maxstatus": svo}

#2. FILE HANDLING (PERSISTENSI DATA)

def simpandata(datalatihan):
    """Menyimpan (Write) data latihan ke file CSV"""
        #try-except digunakan untuk menangani error agar program tidak crash saat penulisan file
    try:
            #with open() digunakan agar file otomatis tertutup setelah selesai digunakan
            #mode='w' (write) digunakan untuk menulis data baru (menimpa file lama)
            #newline='' mencegah adanya baris kosong antar baris data saat file CSV ditulis
        with open(filecsv, mode='w', newline='') as file:
            writer = csv.writer(file)
                #menulis baris header (judul kolom)
                #row mewakili satu baris data berurutan dalam bentuk list yang akan dituliskan ke dalam file CSV
            writer.writerow(["tanggal", "nama", "style", "jenislatihan", "durasimenit", "kalori", "ratahr", "neval"])
                #Gunakan for loop untuk menulis isi list data_latihan ke dalam file
            for d in datalatihan:
                        #d.get digunakan untuk menggantikan kunci atau key jika tidak ditemukan
                writer.writerow([d['tanggal'], d['nama'], d.get('style', '-'), d['jenislatihan'], d['durasimenit'], d['kalori'], d['ratahr'], d.get('neval', '-')])
        print(f"\n✅ Data berhasil disimpan ke {filecsv}")

        #except digunakan untuk menangkap error dari blok try agar program tidak crash
    except Exception as e:
        print(f"\n❌ Terjadi kesalahan saat menyimpan data: {e}")

def muatdata():
    """Memuat (Read) data dari file CSV."""
        #membuat list kosong untuk menampung data dari file
    datalatihan = []
    try:
            #path digunakan mengelola teks jalur suatu file
            #exists digunakan untuk mengecek apakah file di jalur ada atau tidak
            #os.path.exists mengecek apakah file csv sudah ada atau belum
        if os.path.exists(filecsv):
                #mode='r' (read) digunakan untuk membaca isi file
            with open(filecsv, mode='r') as file:
                    #dictreader digunakan untuk membaca baris sebagai dictionary sesuai header
                reader = csv.DictReader(file)
                for row in reader:
                        #append() memasukkan data ke dalam list datalatihan
                        #ubah tipe data kembali ke integer dan float
                    datalatihan.append({"tanggal": row["tanggal"], "nama": row["nama"], "jenislatihan": row["jenislatihan"],"durasimenit": int(row["durasimenit"]),"kalori": float(row["kalori"]),"ratahr": int(row["ratahr"])})
            print(f"\n✅ Berhasil memuat {len(datalatihan)} data latihan dari file")
        else:
            print(f"\n⚠️ File {filecsv} belum tersedia")
    except Exception as e:
        print(f"\n❌ Terjadi kesalahan saat membaca data: {e}")

        #kembalikan list yang sudah berisi data (atau kosong jika gagal/file tidak ada)
    return datalatihan

def eksporlaporan(datalatihan):
    """Ekspor ringkasan performa ke file TXT"""
        #mengecek apakah list datalatihan kosong
    if not datalatihan:
        print("\n⚠️ Tidak ada data untuk diekspor.")
            #return kosong digunakan untuk keluar dari fungsi jika tidak ada data
        return

    try:
            #membuka file txt dengan mode='w' untuk menulis laporan
        with open(filelaporan, mode='w') as file:
            file.write("========================================\n")
            file.write("       LAPORAN RINGKASAN LATIHAN        \n")
            file.write("========================================\n\n")

                #menghitung total durasi dan kalori dari seluruh data di list
            totaldurasi = sum(d['durasimenit'] for d in datalatihan)
            totalkalori = sum(d['kalori'] for d in datalatihan)

                #menulis teks ke dalam file txt menggunakan fungsi .write()
            file.write(f"Total Sesi Latihan : {len(datalatihan)} sesi\n")
            file.write(f"Total Durasi       : {totaldurasi} menit\n")
            file.write(f"Total Kalori Bakar : {totalkalori:.2f} kkal\n\n")

            file.write("Rincian Latihan:\n")
                #menggunakan enumerate untuk membuat nomor urut (i) mulai dari 1
            for i, d in enumerate(datalatihan, 1):
                file.write(f"{i}. {d['tanggal']} | {d['nama']} ({d.get('style','-')}) | {d['jenislatihan']} | {d['durasimenit']}m | Evaluasi: {d.get('neval', '-')}\n")

        print(f"\n✅ Laporan berhasil diekspor ke {filelaporan}")
    except Exception as e:
        print(f"\n❌ Terjadi kesalahan saat mengekspor laporan: {e}")

def tambahdata(datalatihan):
    """Fungsi Append data baru ke database memori"""
    print("\n")
    print("-" * 50)
    print("====Input Data Latihan====")
    print("-" * 50)

        #string khusus digunakan untuk tipe data berupa teks
    tanggal = str(input("Tanggal (YYYY-MM-DD)      : "))
    nama = str(input("Nama Atlet                : "))

    style = str(input("Style (Striker/Grappler)  : ")).lower()
    if style == "striker":
        print("Pilihan: Tendangan / Pukulan / Atletik / Weightlifting")
    else:
        print("Pilihan: Takedown / Submission / Atletik / Weightlifting")

    jenis = str(input("Jenis Latihan             : ")).lower()

    try:
            #integer dan float untuk input berupa angka
        durasi = int(input("Durasi (menit)            : "))

            #Logika Evaluasi berdasarkan input jenis latihan
        neval = "-"

        if jenis in ['tendangan', 'pukulan', 'takedown', 'submission']:
            jml = int(input("Jumlah Gerakan            : "))
            rata = jml / durasi if durasi > 0 else 0
            neval = f"{rata:.1f} gerak/min"
        elif jenis == 'atletik':
            jrk = float(input("Jarak Lari (km)           : "))
            pace = durasi / jrk if jrk > 0 else 0
            neval = f"Pace {pace:.2f}"

        beratkg = float(input("Berat Atlet (kg)          : "))
        intensitas = str(input("Intensitas (ringan/sedang/berat): "))
        ratahr = int(input("Rata-rata HR (bpm)        : "))

            #memanggil fungsi hitungkalori dari modul utilitas
        kalori = hitungkalori(beratkg=beratkg, durasi=durasi, intensitas=intensitas)

            #simpan data ke dalam dictionary sementara
            #round digunakan untuk membulatkan 2 angka di belakang koma
        databaru = {"tanggal": tanggal,"nama": nama,"style": style,"jenislatihan": jenis,"durasimenit": durasi,"kalori": round(kalori, 2), "ratahr": ratahr,"neval": neval}
            #Masukkan dictionary ke dalam list datalatihan
        datalatihan.append(databaru)
        print("\n✅ Data latihan berhasil ditambahkan!")

    except ValueError:
            #Menangkap error jika user menginput huruf pada isian angka
        print("\n❌ Validasi Error: Pastikan durasi, berat, dan HR berupa angka!")

def analisisprogres(datalatihan):
    """Menganalisis progres latihan menggunakan for loop"""
        #mengecek apakah list datalatihan kosong
    if not datalatihan:
        print("\n⚠️ Belum ada data latihan untuk dianalisis.")
        return

    print("\n")
    print("-" * 50)
    print("====Analisis Progres Mingguan====")
    print("-" * 50)
        #{teks:<angka} berarti teks rata kiri dengan lebar spasi tertentu
    print(f"{'Tanggal':<12} | {'Atlet':<10} | {'Durasi':<8} | {'Kalori':<8} | {'Grafik'}")
    print("-" * 50)

    totaldurasi = 0
    totalkalori = 0

        #menggunakan for loop untuk mencetak isi list dan grafis teks
    for data in datalatihan:
        totaldurasi += data['durasimenit']
        totalkalori += data['kalori']

            #membuat grafik sederhana berdasarkan durasi (1 blok = 10 menit)
        blok = int(data['durasimenit'] / 10)
            #menggunakan if inline untuk mencetak blok atau titik kecil jika durasi < 10
        grafik = "█" * blok if blok > 0 else "▪"

        print(f"{data['tanggal']:<12} | {data['nama'][:10]:<10} | {data['durasimenit']:<6} m | {data['kalori']:<8.1f} | {grafik}")

    print("-" * 50)
    print(f"Total Sesi   : {len(datalatihan)}")
    print(f"Total Durasi : {totaldurasi} menit")
    print(f"Total Kalori : {totalkalori:.2f} kkal")
    print("-" * 50)

def caridata(datalatihan):
    """Mencari (Search) data berdasarkan nama atlet."""
        #menggunakan .lower() agar pencarian tidak error terhadap huruf besar/kecil
    namacari = str(input("\nMasukkan nama atlet yang dicari: ")).lower()
        #variabel boolean untuk penanda apakah data ketemu atau tidak
    ditemukan = False

    print("\n")
    print("-" * 50)
    print("====Hasil Pencarian====")
    print("-" * 50)

    for d in datalatihan:
            #jika namacari ada di dalam nama atlet di database
        if namacari in d['nama'].lower():
            print(f"{d['tanggal']} | {d['nama']} ({d.get('style','-')}) | {d['jenislatihan']} | Durasi: {d['durasimenit']}m | Evaluasi: {d.get('neval','-')}")
                #ubah status menjadi True karena data ditemukan
            ditemukan = True

        #jika loop selesai dan ditemukan masih False
    if not ditemukan:
        print("Data atlet tidak ditemukan")

#3. APLIKASI UTAMA (MAIN LOOP & MENU)

    #main() digunakan sebagai fungsi utama yang menjadi titik awal eksekusi seluruh alur program
def main():
        #buat list kosong sebagai database utama sementara saat program berjalan
    databaselatihan = []

        #menggunakan while True untuk membuat aplikasi berjalan terus-menerus sampai diketik 0
    while True:
        print("\n\n")
        print("=" * 40)
        print(" APLIKASI MANAJEMEN PERFORMA ATLET ")
        print(" Rekayasa Keolahragaan ITERA 2026")
        print("=" * 40)
        print("[1] 📝 Registrasi & Zona Atlet")
        print("[2] 🏋‍♂️ Input Data Latihan")
        print("[3] ❤️‍🩹 Evaluasi Kondisi Fisik")
        print("[4] 📊 Analisis Progres Mingguan")
        print("[5] 🔍 Cari Data Atlet")
        print("[6] 💾 Simpan Data ke File")
        print("[7] 📂 Muat Data dari File")
        print("[8] 📋 Ekspor Laporan Ringkasan")
        print("[0] 🚪 Keluar")
        print("=" * 40)

            #input berupa string untuk mengecek teks 1, 2, dll
        pilihan = str(input("Pilih menu (0-6): "))

        if pilihan == '1':
            print("\n")
            print("-" * 50)
            print("====Registrasi Atlet====")
            print("-" * 50)
            try:
                nama = str(input("Nama        : "))
                usia = int(input("Usia        : "))
                beratkg = float(input("Berat (kg)  : "))
                tinggicm = float(input("Tinggi (cm) : "))

                    #memanggil fungsi-fungsi dari modul utilitas
                imt = hitungimt(beratkg, tinggicm)
                hrmax = hitunghrmax(usia)
                zona = zonalatihan(hrmax)

                print("\n=== Ringkasan Registrasi ===")
                print(f"Nama     : {nama}")
                    #:.2f bermaksud membuat angka di belakang koma menjadi 2 angka
                print(f"IMT      : {imt:.2f} ({kimt(imt)})")
                print(f"HRmax    : {hrmax} bpm")
                print("Zona Latihan:")
                print("Zona Latihan:")
                print(f" - Ringan: {zona['ringan']:.1f} bpm")
                print("   ==> Saran: Recovery, pemanasan, atau drill teknik dasar.")
                print(f" - Sedang: {zona['sedang']:.1f} bpm")
                print("   ==> Saran: Ketahanan kardio, pad work, atau sparring ringan.")
                print(f" - Berat : {zona['berat']:.1f} bpm")
                print("   ==> Saran: Peningkatan VO2Max, power/kecepatan, atau sparring intens.")
            except valueerror:
                print("❌ Input tidak valid. Usia, berat, dan tinggi harus berupa angka.")

        elif pilihan == '2':
                #memanggil fungsi input dan kirim list databaselatihan
            tambahdata(databaselatihan)

        elif pilihan == '3':
            print("\n")
            print("-" * 50)
            print("====Evaluasi Kondisi Fisik====")
            print("-" * 50)
            try:
                hr = int(input("Detak Jantung Istirahat (bpm) : "))
                sis = int(input("Tekanan Darah Sistolik        : "))
                dias = int(input("Tekanan Darah Diastolik       : "))
                vo2 = float(input("VO2Max                        : "))
                imtval = float(input("IMT                           : "))

                    #memanggil fungsi evaluasi performa dari utilitas
                hasil = evaluasiperforma(hr, sis, dias, vo2, imtval)
                print("\n===Hasil Evaluasi===")
                print(f"Detak Jantung: {hasil['hrstatus']}")
                print(f"Tekanan Darah: {hasil['tdstatus']}")
                print(f"VO2Max       : {hasil['vo2maxstatus']}")
            except valueerror:
                print("❌ Input harus berupa angka valid")

        elif pilihan == '4':
                #memanggil fitur analisis dari menu 4
            analisisprogres(databaselatihan)

        elif pilihan == '5':
            caridata(databaselatihan)

        elif pilihan == '6':
                #mengecek apakah list/database memori kosong
            if not databaselatihan:
                print("\n⚠️ Belum ada data di memori untuk disimpan.")
            else:
                simpandata(databaselatihan)

        elif pilihan == '7':
                #mengganti isi list dengan data yang dikembalikan dari file CSV
            databaselatihan = muatdata()

        elif pilihan == '8':
                #memanggil fitur ekspor dari menu 8
            eksporlaporan(databaselatihan)

        elif pilihan == '0':
            print("\nTerima kasih telah menggunakan aplikasi ini")
                #break digunakan untuk menghentikan loop secara paksa sehingga aplikasi tertutup
            break

        else:
            print("\n❌ Pilihan menu tidak valid. Silakan coba lagi.")

    #Memastikan fungsi main() dijalankan jika file ini dieksekusi secara langsung
    #if __name__ == "__main__": main() digunakan untuk memastikan bahwa sebuah file Python hanya akan menjalankan kode di dalamnya
if __name__ == "__main__":
    main()




 APLIKASI MANAJEMEN PERFORMA ATLET 
 Rekayasa Keolahragaan ITERA 2026
[1] 📝 Registrasi & Zona Atlet
[2] 🏋‍♂️ Input Data Latihan
[3] ❤️‍🩹 Evaluasi Kondisi Fisik
[4] 📊 Analisis Progres Mingguan
[5] 🔍 Cari Data Atlet
[6] 💾 Simpan Data ke File
[7] 📂 Muat Data dari File
[8] 📋 Ekspor Laporan Ringkasan
[0] 🚪 Keluar
Pilih menu (0-6): 1


--------------------------------------------------
====Registrasi Atlet====
--------------------------------------------------
Nama        : Khabib
Usia        : 38
Berat (kg)  : 80
Tinggi (cm) : 177

=== Ringkasan Registrasi ===
Nama     : Khabib
IMT      : 25.54 (Overweight)
HRmax    : 182 bpm
Zona Latihan:
Zona Latihan:
 - Ringan: 91.0 bpm
   ↳ Saran: Recovery, pemanasan, atau drill teknik dasar.
 - Sedang: 127.4 bpm
   ↳ Saran: Ketahanan kardio, pad work, atau sparring ringan.
 - Berat : 154.7 bpm
   ↳ Saran: Peningkatan VO2Max, power/kecepatan, atau sparring intens.



 APLIKASI MANAJEMEN PERFORMA ATLET 
 Rekayasa Keolahragaan ITERA 2026
[1] 📝 Registra